# F1 strategy agent - evaluation

1. **Offline evaluation** (no API needed): RAW policy vs SAFE (policy + safety layer) accuracy against the real drivers' decisions.
2. **API demo**: start the API first with `uvicorn api:app --reload`, then run the two API cells. They send random states and only check the API mechanics.

In [ ]:
# Offline evaluation of the model + safety layer (RAW vs SAFE accuracy) without the API
import numpy as np
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

from utils import selected_features, scale_observation
from safety_layer import verify_action, RaceState

# === Load the model and data ===
# v6_best is the checkpoint with the best reward/step during training
# (recommended over the final model, which can degrade due to PPO
# instability late in training).
model = PPO.load("saved_models/ppo_f1_rl_logic_v6_best.zip")

obs_array = np.load("data/obs_array.npy")
true_actions = np.load("data/true_actions.npy")

raw_actions = []
safe_actions = []

viol_before = 0
overrides_count = 0

pit_overrides = 0
comp_overrides = 0
style_overrides = 0

# === obs_array is a CONCATENATION of several real races, not one long race ===
# Segmentation by a drop in LapNumber (same principle as predict_strategies.py);
# without it, R10 (two mandatory dry compounds) would only check the last 3 rows
# of the WHOLE array instead of the end of every race.
i_lap = selected_features.index("LapNumber")
i_weather = selected_features.index("Weather")

lap_numbers = obs_array[:, i_lap]
race_start_indices = [0]
for idx in range(1, len(lap_numbers)):
    if lap_numbers[idx] < lap_numbers[idx - 1]:
        race_start_indices.append(idx)
race_start_indices.append(len(obs_array))

n_races = len(race_start_indices) - 1
print(f"[Segmentation] Detected {n_races} separate races/stints in obs_array")

race_state = None
seg_end = 0

for row_idx, obs in enumerate(obs_array):
    if row_idx in race_start_indices[:-1]:
        pos = race_start_indices.index(row_idx)
        seg_end = race_start_indices[pos + 1]
        race_state = RaceState(total_laps=seg_end - row_idx, force_window=3)

    # The model was trained on SCALED observations (utils.scale_observation),
    # so it must receive the same input here.
    action, _ = model.predict(scale_observation(obs).reshape(1, -1), deterministic=True)
    raw = action[0] if isinstance(action[0], (list, np.ndarray)) else action
    raw = np.array(raw, dtype=np.int32)

    # verify_action receives the RAW (unscaled) state, because its thresholds
    # (tyre_life < 2.0, front_gap < 1.5, etc.) are in real units.
    safe, report = verify_action(obs, raw, selected_features, race_state=race_state)

    raw_actions.append(raw)
    safe_actions.append(safe)

    if report.violations_before:
        viol_before += 1
    if report.overrides:
        overrides_count += 1

    for o in report.overrides:
        if o["field"] == "pit":
            pit_overrides += 1
        elif o["field"] == "compound":
            comp_overrides += 1
        elif o["field"] == "style":
            style_overrides += 1

    race_state.record(weather=int(round(float(obs[i_weather]))), compound=int(safe[1]))

raw_actions = np.array(raw_actions, dtype=np.int32)
safe_actions = np.array(safe_actions, dtype=np.int32)

# === Accuracy RAW vs SAFE ===
acc_pit_raw = accuracy_score(true_actions[:, 0], raw_actions[:, 0])
acc_comp_raw = accuracy_score(true_actions[:, 1], raw_actions[:, 1])
acc_style_raw = accuracy_score(true_actions[:, 2], raw_actions[:, 2])

acc_pit_safe = accuracy_score(true_actions[:, 0], safe_actions[:, 0])
acc_comp_safe = accuracy_score(true_actions[:, 1], safe_actions[:, 1])
acc_style_safe = accuracy_score(true_actions[:, 2], safe_actions[:, 2])

n = len(obs_array)

print(f"Safety layer override rate: {overrides_count/n:.2%}" if n else "No samples.")
print(f"Violations (raw): {viol_before/n:.2%}" if n else "No samples.")
print(f"Overrides by field: pit={pit_overrides}, compound={comp_overrides}, style={style_overrides}")

print("\nPrediction accuracy (RAW = without the safety layer):")
print(f"Pit stop:     {acc_pit_raw:.2%}")
print(f"Compound:     {acc_comp_raw:.2%}")
print(f"Driving style:{acc_style_raw:.2%}")

print("\nPrediction accuracy (SAFE = with the safety layer):")
print(f"Pit stop:     {acc_pit_safe:.2%}")
print(f"Compound:     {acc_comp_safe:.2%}")
print(f"Driving style:{acc_style_safe:.2%}")

# === Accuracy plot ===
labels = ["Pit", "Compound", "Style"]
raw_vals = [acc_pit_raw, acc_comp_raw, acc_style_raw]
safe_vals = [acc_pit_safe, acc_comp_safe, acc_style_safe]

x = np.arange(len(labels))
w = 0.35

plt.figure(figsize=(7,4))
plt.bar(x - w/2, raw_vals, width=w, label="RAW")
plt.bar(x + w/2, safe_vals, width=w, label="SAFE")
plt.xticks(x, labels)
plt.ylim(0, 1)
plt.title("Agent accuracy per action: RAW vs SAFE")
plt.ylabel("Accuracy")
plt.grid(axis="y")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# === main.ipynb - interactive notebook for testing the F1 RL agent and the safety layer ===

# Start the FastAPI server in a separate terminal first:
# uvicorn api:app --reload

# If a package is missing:
# pip install requests numpy matplotlib

# Run from the project root:
# python -m uvicorn api:app --reload --host 127.0.0.1 --port 8000

import requests
import numpy as np
import matplotlib.pyplot as plt
import time
from utils import selected_features  # defines the correct state dimension

# REST API endpoint
url = "http://127.0.0.1:8000/predict_strategy"
N_FEATURES = len(selected_features)

def simulate_episode(n_steps=20, verbose=True):
    rewards = []
    safe_actions = []
    raw_actions = []
    override_flags = []  # 1 if the safety layer changed something, 0 otherwise

    # Dummy state with the correct dimension
    state = np.random.rand(N_FEATURES).tolist()

    for step in range(n_steps):
        response = requests.post(url, json={"state": state})
        if response.status_code != 200:
            raise RuntimeError(f"API error {response.status_code}: {response.text}")

        data = response.json()

        # The API returns "action" = SAFE action (after the safety layer) + "explanation"
        safe_action = data["action"]
        explanation = data.get("explanation", {})

        raw_action = explanation.get("raw_action", None)
        overrides = explanation.get("logic", {}).get("overrides", [])
        was_overridden = 1 if overrides else 0

        # Dummy reward (demo only)
        reward = float(np.sum(safe_action)) - 0.1 * np.random.rand()

        rewards.append(reward)
        safe_actions.append(safe_action)
        raw_actions.append(raw_action if raw_action is not None else [-1, -1, -1])
        override_flags.append(was_overridden)

        if verbose:
            print(f"\nStep {step+1}:")
            print(f"State (first 8): {np.round(state[:8], 2)} ...")
            print(f"RAW action:  {raw_action} | {explanation.get('raw_human')}")
            print(f"SAFE action: {safe_action} | {explanation.get('safe_human')}")
            print(f"Overrides: {len(overrides)} | Violations: {len(explanation.get('logic', {}).get('violations_before', []))}")
            if overrides:
                for o in overrides:
                    print(f" - {o['field']}: {o['from']} -> {o['to']} ({o['rule']})")
            print(f"Reward: {reward:.2f}")
            print("-" * 60)

        # Update the state (dummy dynamics, correct dimension)
        state = (np.random.rand(N_FEATURES) * (0.5 + reward / 10)).tolist()
        time.sleep(0.05)

    return rewards, np.array(raw_actions), np.array(safe_actions), np.array(override_flags)

# Run the simulation
rewards, raw_actions, safe_actions, override_flags = simulate_episode(n_steps=20, verbose=True)

# === Plot: reward per step ===
plt.figure(figsize=(10, 4))
plt.plot(rewards, marker='o')
plt.title("Simulated reward (SAFE action) per step")
plt.xlabel("Step")
plt.ylabel("Reward")
plt.grid(True)
plt.tight_layout()
plt.show()

print("\nOverride rate in the episode:", override_flags.mean())


In [ ]:
# API smoke tests (single + batch + explain) with the correct dimension and readable output
import requests
import numpy as np
from utils import selected_features

BASE_URL = "http://127.0.0.1:8000"
N_FEATURES = len(selected_features)

# NOTE: these are RANDOM values in [0, 1] for a quick smoke test of the API
# mechanics (does the endpoint work, does the safety layer fire). They do NOT
# represent a realistic race state (e.g. LapNumber should be an integer 1-70,
# and TyreLife < 2 makes rule R4 force a pit on almost every call). For the
# real model evaluation use predict_strategies.py or the offline cell.
def generate_dummy_state(n=N_FEATURES):
    return (np.random.rand(n)).tolist()

# === 1. predict_strategy ===
def test_predict_strategy():
    url = f"{BASE_URL}/predict_strategy"
    state = generate_dummy_state()
    response = requests.post(url, json={"state": state})
    print("=== /predict_strategy ===")
    if response.status_code != 200:
        print("ERROR:", response.status_code, response.text)
        return None

    data = response.json()
    exp = data.get("explanation", {})

    print("SAFE action:", data.get("action"), "|", exp.get("safe_human"))
    print("RAW action :", exp.get("raw_action"), "|", exp.get("raw_human"))

    logic = exp.get("logic", {})
    print("Overrides:", len(logic.get("overrides", [])))
    print("Violations before:", len(logic.get("violations_before", [])))
    if logic.get("overrides"):
        for o in logic["overrides"]:
            print(f" - {o['field']}: {o['from']} -> {o['to']} ({o['rule']})")

    return data

# === 2. predict_strategy_batch ===
def test_predict_strategy_batch(batch_size=5):
    url = f"{BASE_URL}/predict_strategy_batch"
    states = [{"state": generate_dummy_state()} for _ in range(batch_size)]
    response = requests.post(url, json=states)

    print("\n=== /predict_strategy_batch ===")
    if response.status_code != 200:
        print("ERROR:", response.status_code, response.text)
        return None

    data = response.json()
    summary = data.get("summary", {})
    print("SUMMARY:", summary)

    # show the first result
    results = data.get("results", [])
    if results:
        first = results[0]
        print("\nFirst result:")
        print("RAW:", first.get("raw_action"), "|", first.get("raw_human"))
        print("SAFE:", first.get("safe_action"), "|", first.get("safe_human"))
        print("Logic:", first.get("logic"))

    return data

# === 3. explain_strategy ===
def test_explain_strategy():
    url = f"{BASE_URL}/explain_strategy"
    state = generate_dummy_state()
    response = requests.post(url, json={"state": state})

    print("\n=== /explain_strategy ===")
    if response.status_code != 200:
        print("ERROR:", response.status_code, response.text)
        return None

    print("Response:", response.json())
    return response.json()

# === Run the tests ===
predict_result = test_predict_strategy()
batch_result = test_predict_strategy_batch(batch_size=8)
explanation_result = test_explain_strategy()